# DATALUS OAA Compliance: Empirical Privacy & Utility Audit

This notebook demonstrates the **Autonomous Audit Orchestrator (OAA)**. 
We evaluate the generated synthetic data against rigorous privacy and utility benchmarks to evaluate its safety for public release 
under data protection regulations (e.g., LGPD/GDPR).

The audit includes **Membership Inference Attack (MIA)** simulations and **Distance to Closest Record (DCR)** checks for memorization.

In [ ]:
# ============================================================
# DATASET CONFIGURATION
# ============================================================
# Set USE_REAL_DATA to True to download real SIH-SUS records
# from the DATASUS FTP servers. Set to False to generate a
# fully configurable synthetic dataset.
USE_REAL_DATA = False

import polars as pl
import numpy as np

SYNTHETIC_CONFIG = {
    'n_rows': 100_000,
    'seed': 42,
    'columns': {
        'MORTE': {
            'dtype': pl.Int64,
            'generator': lambda n, rng: rng.choice([0, 1], n, p=[0.92, 0.08]).astype(np.int64),
        },
        'IDADE': {
            'dtype': pl.Float64,
            'generator': lambda n, rng: rng.integers(0, 100, n).astype(np.float64),
        },
        'SEXO': {
            'dtype': pl.Utf8,
            'generator': lambda n, rng: rng.choice(['0', '1'], n),
        },
        'DIAS_PERMANENCIA': {
            'dtype': pl.Float64,
            'generator': lambda n, rng: np.clip(rng.poisson(5, n), 0, 100).astype(np.float64),
        },
        'VALOR_TOTAL': {
            'dtype': pl.Float64,
            'generator': lambda n, rng: np.round(np.exp(rng.normal(7, 1.5, n)), 2),
        },
        'PROCEDIMENTO_PRINCIPAL': {
            'dtype': pl.Utf8,
            'generator': lambda n, rng: rng.choice(['AB', 'CD', 'EF', 'GH', 'IJ'], n),
        },
        'MUNICIPIO_MOV': {
            'dtype': pl.Utf8,
            'generator': lambda n, rng: rng.choice(['3550308', '3304557', '3106200'], n),
        },
    },
    'target_column': 'MORTE',
}

## 1. Environment Setup & Data Verification
Detecting environment and verifying that real and synthetic datasets are ready for audit. 
If data is missing, we trigger a fallback generation process.

**Google Colab:** After the installation cell below finishes, Google Colab may automatically restart the runtime due to dependency changes. If this occurs, or if you see the installation complete message, manually restart the runtime via `Runtime > Restart and run all` and re-execute all cells from the first cell. The `.installed` flag will prevent re-installation on the next run.

**WORKING_DIR:** If you encounter a "No such file or directory" error in any `datalus` command, the `WORKING_DIR` variable may not resolve correctly. In that case, manually set `WORKING_DIR` to the absolute path of your working directory in the environment setup cell below.

In [ ]:
import os
import sys
from pathlib import Path

def get_working_dir():
    if 'google.colab' in sys.modules:
        return '/content/datalus_audit'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return '/kaggle/working/datalus_audit'
    return './datalus_audit'

WORKING_DIR = get_working_dir()
os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Working directory: {WORKING_DIR}')

# Verify WORKING_DIR is writable
try:
    test_file = Path(WORKING_DIR) / '.write_test'
    test_file.touch()
    test_file.unlink()
except (OSError, PermissionError):
    print(f'WARNING: Cannot write to {WORKING_DIR}')
    print('Please set WORKING_DIR manually:')
    print('  WORKING_DIR = "/path/to/your/directory"')

INSTALLED = Path(WORKING_DIR) / '.installed'

if not INSTALLED.exists():
    if 'google.colab' in sys.modules or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        !git clone --depth 1 https://github.com/emanuellcs/datalus.git "{WORKING_DIR}/source" 2>/dev/null || true
        %cd "{WORKING_DIR}/source"
        !pip install --quiet -e '.[dev]' pysus polars matplotlib seaborn scikit-learn nest_asyncio 2>&1 | tail -5
        sys.path.insert(0, f'{WORKING_DIR}/source')
        INSTALLED.touch()
        print()
        print('=' * 60)
        print('  INSTALLATION COMPLETE')
        print('  Please restart the runtime manually:')
        print('    Runtime > Restart and run all')
        print('  Then re-execute all cells from the beginning.')
        print('=' * 60)
    else:
        INSTALLED.touch()
        print('Local environment. Dependencies assumed installed.')
else:
    src = Path(WORKING_DIR) / 'source'
    if src.is_dir():
        sys.path.insert(0, str(src))
    print('Dependencies already installed.')

real_path = Path(f'{WORKING_DIR}/processed.parquet')
synth_path = Path(f'{WORKING_DIR}/synthetic.parquet')
schema_path = Path(f'{WORKING_DIR}/schema_config.json')

if not (real_path.exists() and synth_path.exists()):
    print('Data not found. Generating fallback dummy datasets for audit demo...')
    from pysus import sih, sim, sinan
    from datetime import datetime
    import urllib.request
    import asyncio

    STATES = ['SP', 'RJ', 'MG', 'PR', 'RS', 'BA', 'DF']

    def fmt_tier(label, state, year, month, count):
        return f'Tier {label}: {state}/{year}/{month} ({count} records)'

    def try_pysus():
        now = datetime.now()
        sy, sm = (now.year, now.month - 3) if now.month > 3 else (now.year - 1, now.month + 9)
        for state in STATES:
            for y in range(sy, sy - 2, -1):
                for m in range(sm if y == sy else 12, 0, -1):
                    try:
                        df = sih(state=state, year=y, month=[m], group='RD', as_dataframe=True)
                        if df is not None and len(df) > 0 and 'MORTE' in df.columns:
                            print(fmt_tier('1: SIH-RD', state, y, m, len(df)))
                            return pl.from_pandas(df)
                    except Exception:
                        continue
            for y in range(sy, sy - 2, -1):
                try:
                    df = sim(state=state, year=y, as_dataframe=True)
                    if df is not None and len(df) > 0:
                        print(fmt_tier('2: SIM', state, y, 'all', len(df)))
                        df['MORTE'] = 1
                        return pl.from_pandas(df)
                except Exception:
                    continue
        for y in range(sy, sy - 2, -1):
            try:
                df = sinan(disease='deng', year=y, as_dataframe=True)
                if df is not None and len(df) > 0:
                    print(fmt_tier('3: SINAN-deng', 'BR', y, 'all', len(df)))
                    if 'MORTE' not in df.columns:
                        df['MORTE'] = 0
                    return pl.from_pandas(df)
            except Exception:
                continue
        return None

    def try_direct_ftp():
        try:
            import nest_asyncio
            nest_asyncio.apply()
        except ImportError:
            return None
        from pysus.api.extensions import DBC
        now = datetime.now()
        sy, sm = (now.year, now.month - 3) if now.month > 3 else (now.year - 1, now.month + 9)
        base = 'ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/199301_/Dados/'
        for state in ['SP', 'RJ', 'MG']:
            for y in range(sy, sy - 2, -1):
                for m in range(sm if y == sy else 12, 0, -1):
                    fname = f'RD{state}{y % 100:02d}{m:02d}.dbc'
                    try:
                        with urllib.request.urlopen(base + fname, timeout=30) as r:
                            dbc_path = Path('/tmp') / fname
                            dbc_path.write_bytes(r.read())
                        df = asyncio.run(DBC(path=dbc_path).load())
                        if df is not None and len(df) > 0:
                            print(fmt_tier('4: FTP direct', state, y, m, len(df)))
                            return pl.from_pandas(df)
                    except Exception:
                        continue
        return None

    if USE_REAL_DATA:
        print('Downloading clinical data from DATASUS...')
        df = try_pysus()
        if df is None:
            df = try_direct_ftp()
        if df is None:
            raise RuntimeError(
                'Could not download real data from DATASUS. '
                'Set USE_REAL_DATA = False in the configuration cell '
                'to use a synthetic dataset instead.'
            )
    else:
        print(f'Generating synthetic dataset ({SYNTHETIC_CONFIG["n_rows"]} rows)...')
        rng = np.random.default_rng(SYNTHETIC_CONFIG['seed'])
        n = SYNTHETIC_CONFIG['n_rows']
        data = {}
        for name, cfg in SYNTHETIC_CONFIG['columns'].items():
            data[name] = cfg['generator'](n, rng)
        df = pl.DataFrame(
            data,
            schema={name: cfg['dtype'] for name, cfg in SYNTHETIC_CONFIG['columns'].items()},
        )

    df = df.with_columns(
        pl.col('MORTE').cast(pl.Int64).fill_null(0)
    )

    cols_to_keep = [
        c for c in df.columns
        if not c.startswith(('N_AIH', 'IDENT', 'CNS', 'CPF', 'CNPJ'))
    ]
    df = df.select(
        ['MORTE'] + [c for c in cols_to_keep if c != 'MORTE']
    ).head(2000)
    df.write_parquet(f'{WORKING_DIR}/raw_sample.parquet')
    
    !datalus ingest {WORKING_DIR}/raw_sample.parquet {WORKING_DIR}/processed.parquet --schema-path {WORKING_DIR}/schema_config.json --target-column MORTE
    !datalus train {WORKING_DIR}/schema_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/artifacts --epochs 1 --batch-size 256 --max-steps 1000 --gpu 0
    !datalus sample {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet --n-records 2000 --ddim-steps 50 --seed 42 --cfg-scale 1.0
else:
    print('Verified: Audit datasets are ready.')


## 2. Executing the Autonomous Audit Orchestrator (OAA)
We run the `datalus audit` command. For this demonstration, we use a limited subset of rows to ensure fast execution. 
The `--mia-mode release` flag triggers the full shadow-model attack simulation.

In [ ]:
!datalus audit {WORKING_DIR}/processed.parquet {WORKING_DIR}/synthetic.parquet {WORKING_DIR}/schema_config.json {WORKING_DIR}/audit_report.json --target-column MORTE --mia-mode release --max-audit-rows 1000

## 3. Parsing and Interpreting the Audit Results
Loading the JSON report and extracting the high-level Pass/Fail verdict based on privacy and utility thresholds.

In [ ]:
import json
with open(f'{WORKING_DIR}/audit_report.json', 'r') as f:
    report = json.load(f)

verdict = "PASSED" if report.get('overall_verdict', False) else "FAILED"
print(f"--- AUDIT VERDICT: {verdict} ---")
print(f"Privacy Score: {report.get('privacy', {}).get('mia_roc_auc', 'N/A')}")
print(f"Utility MLE Ratio: {report.get('utility', {}).get('mle_ratio', 'N/A')}")


## 4. Privacy Validation: Distance to Closest Record (DCR)
DCR measures if the model has memorized specific real records. We check the 5th percentile distance against the alert threshold.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'privacy' in report and 'dcr_stats' in report['privacy']:
    dcr = report['privacy']['dcr_stats']
    p5_dcr = dcr.get('p5', 0)
    threshold = 0.05 # Example threshold
    
    plt.figure(figsize=(10, 5))
    colors = ['green' if p5_dcr > threshold else 'red']
    plt.bar(['5th Percentile DCR'], [p5_dcr], color=colors)
    plt.axhline(y=threshold, color='black', linestyle='--', label='Security Threshold')
    plt.title('Privacy Check: Distance to Closest Record (Memorization Risk)')
    plt.ylabel('Normalized Distance')
    plt.legend()
    plt.show()


## 5. Privacy Validation: Shadow-MIA Attack AUC
An AUC close to 0.50 indicates that the attacker cannot distinguish between members and non-members of the training set.

In [ ]:
if 'privacy' in report and 'mia_roc_auc' in report['privacy']:
    auc = report['privacy']['mia_roc_auc']
    
    plt.figure(figsize=(10, 5))
    plt.bar(['MIA ROC-AUC'], [auc], color='orange')
    plt.axhline(y=0.5, color='blue', linestyle='--', label='Random Guess (Perfect Privacy)')
    plt.axhline(y=0.6, color='red', linestyle='--', label='Alert Threshold')
    plt.ylim(0.4, 1.0)
    plt.title('Privacy Check: Membership Inference Attack Resistance')
    plt.ylabel('ROC-AUC Score')
    plt.legend()
    plt.show()


## 6. Utility Validation: TSTR vs TRTR Machine Learning Efficacy
The MLE ratio proves that models trained on synthetic data perform nearly as well as those trained on real data.

In [ ]:
if 'utility' in report and 'mle_scores' in report['utility']:
    mle = report['utility']['mle_scores']
    labels = ['TSTR', 'TRTR']
    scores = [mle.get('tstr', 0), mle.get('trtr', 0)]
    
    plt.figure(figsize=(10, 5))
    sns.barplot(x=labels, y=scores, palette='viridis')
    plt.title(f"Utility Check: MLE Ratio ({report['utility'].get('mle_ratio', 0):.2f})")
    plt.ylabel('F1 / Accuracy Score')
    plt.show()
